In [ ]:
!pip -q install requests pandas tqdm

import pandas as pd
import requests
import io
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.colab import files

ANO_INICIAL = 2025
ANO_FINAL = pd.Timestamp.today().year
DATA_HOJE = pd.Timestamp.today().normalize()
MAX_WORKERS = 8

ARQUIVO_FINAL = "Base_Dengue_Brasil_2025_ate_hoje.csv"
ARQUIVO_ERROS = "Municipios_com_erro_InfoDengue.csv"

API_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
API_INFODENGUE = "https://info.dengue.mat.br/api/alertcity"

teste_params = {
    "geocode": 3304557,
    "disease": "dengue",
    "format": "csv",
    "ew_start": 1,
    "ew_end": 53,
    "ey_start": 2025,
    "ey_end": 2025
}

try:
    teste = requests.get(
        API_INFODENGUE,
        params=teste_params,
        timeout=60
    )
    print("Status da API:", teste.status_code)
except Exception as erro:
    print("Erro:", erro)

resposta_ibge = requests.get(
    API_IBGE,
    timeout=60
)

resposta_ibge.raise_for_status()

municipios_json = resposta_ibge.json()

municipios = []

for municipio in municipios_json:
    try:
        uf = municipio["microrregiao"]["mesorregiao"]["UF"]

        municipios.append({
            "geocode": int(municipio["id"]),
            "municipio": municipio["nome"],
            "UF": uf["sigla"]
        })

    except Exception:
        pass

df_municipios = pd.DataFrame(municipios)

df_municipios = df_municipios.drop_duplicates(
    subset=["geocode"]
)

print(
    "Municípios encontrados:",
    len(df_municipios)
)

def consultar_municipio(registro):

    geocode = registro["geocode"]
    municipio = registro["municipio"]
    uf = registro["UF"]

    resultados = []
    erros_local = []

    for ano in range(
        ANO_INICIAL,
        ANO_FINAL + 1
    ):

        parametros = {
            "geocode": geocode,
            "disease": "dengue",
            "format": "csv",
            "ew_start": 1,
            "ew_end": 53,
            "ey_start": ano,
            "ey_end": ano
        }

        try:

            resposta = requests.get(
                API_INFODENGUE,
                params=parametros,
                timeout=90
            )

            if resposta.status_code != 200:
                erros_local.append(
                    f"Ano {ano}: HTTP {resposta.status_code}"
                )
                continue

            texto = resposta.text.strip()

            if not texto:
                continue

            if texto.startswith("[EE]"):
                erros_local.append(
                    f"Ano {ano}: {texto}"
                )
                continue

            try:
                dados = pd.read_csv(
                    io.StringIO(texto)
                )
            except Exception as erro_csv:
                erros_local.append(
                    f"Ano {ano}: {erro_csv}"
                )
                continue

            if dados.empty:
                continue

            dados["geocode"] = geocode
            dados["municipio_nome"] = municipio
            dados["UF"] = uf

            if "data_iniSE" in dados.columns:

                dados["data_iniSE"] = pd.to_datetime(
                    dados["data_iniSE"],
                    errors="coerce"
                )

                dados = dados[
                    dados["data_iniSE"].dt.year >= ANO_INICIAL
                ]

                dados = dados[
                    dados["data_iniSE"] <= DATA_HOJE
                ]

            if not dados.empty:
                resultados.append(dados)

        except Exception as erro:
            erros_local.append(
                f"Ano {ano}: {repr(erro)}"
            )

    if len(resultados) == 0:

        return {
            "status": "sem_dados",
            "geocode": geocode,
            "municipio": municipio,
            "UF": uf,
            "dados": None,
            "erros": erros_local
        }

    try:

        dados_final = pd.concat(
            resultados,
            ignore_index=True
        )

    except Exception as erro:

        return {
            "status": "erro",
            "geocode": geocode,
            "municipio": municipio,
            "UF": uf,
            "dados": None,
            "erros": [repr(erro)]
        }

    return {
        "status": "ok",
        "geocode": geocode,
        "municipio": municipio,
        "UF": uf,
        "dados": dados_final,
        "erros": erros_local
    }

inicio = time.time()

dados_finais = []
municipios_sem_dados = []
municipios_com_erro = []

lista_municipios = df_municipios.to_dict(
    "records"
)

total_municipios = len(
    lista_municipios
)

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    tarefas = {
        executor.submit(
            consultar_municipio,
            municipio
        ): municipio

        for municipio in lista_municipios
    }

    for tarefa in tqdm(
        as_completed(tarefas),
        total=total_municipios,
        desc="Consultando municípios"
    ):

        try:

            resultado = tarefa.result()

        except Exception as erro:

            municipio = tarefas[tarefa]

            municipios_com_erro.append({
                "geocode": municipio["geocode"],
                "municipio": municipio["municipio"],
                "UF": municipio["UF"],
                "erro": repr(erro)
            })

            continue

        if resultado["status"] == "ok":

            dados_finais.append(
                resultado["dados"]
            )

        elif resultado["status"] == "sem_dados":

            municipios_sem_dados.append({
                "geocode": resultado["geocode"],
                "municipio": resultado["municipio"],
                "UF": resultado["UF"],
                "erro": "Nenhum dado retornado"
            })

        else:

            municipios_com_erro.append({
                "geocode": resultado["geocode"],
                "municipio": resultado["municipio"],
                "UF": resultado["UF"],
                "erro": " | ".join(
                    resultado.get(
                        "erros",
                        []
                    )
                )
            })

if len(dados_finais) == 0:

    raise Exception(
        "A API não retornou dados."
    )

df_brasil = pd.concat(
    dados_finais,
    ignore_index=True
)

if "data_iniSE" in df_brasil.columns:

    df_brasil["data_iniSE"] = pd.to_datetime(
        df_brasil["data_iniSE"],
        errors="coerce"
    )

    df_brasil = df_brasil[
        df_brasil["data_iniSE"].notna()
    ]

    df_brasil = df_brasil[
        df_brasil["data_iniSE"] <= DATA_HOJE
    ]

colunas_duplicidade = []

if "geocode" in df_brasil.columns:
    colunas_duplicidade.append("geocode")

if "SE" in df_brasil.columns:
    colunas_duplicidade.append("SE")

if len(colunas_duplicidade) > 0:

    df_brasil = df_brasil.drop_duplicates(
        subset=colunas_duplicidade
    )

colunas_ordenacao = []

if "UF" in df_brasil.columns:
    colunas_ordenacao.append("UF")

if "municipio_nome" in df_brasil.columns:
    colunas_ordenacao.append("municipio_nome")

if "data_iniSE" in df_brasil.columns:
    colunas_ordenacao.append("data_iniSE")

if len(colunas_ordenacao) > 0:

    df_brasil = df_brasil.sort_values(
        colunas_ordenacao,
        kind="stable"
    )

colunas_inicio = [
    "UF",
    "municipio_nome",
    "geocode",
    "data_iniSE",
    "SE"
]

colunas_existentes = [
    coluna
    for coluna in colunas_inicio
    if coluna in df_brasil.columns
]

outras_colunas = [
    coluna
    for coluna in df_brasil.columns
    if coluna not in colunas_existentes
]

df_brasil = df_brasil[
    colunas_existentes + outras_colunas
]

if "data_iniSE" in df_brasil.columns:

    df_brasil["data_iniSE"] = (
        df_brasil["data_iniSE"]
        .dt.strftime("%d/%m/%Y")
    )

df_brasil.to_csv(
    ARQUIVO_FINAL,
    index=False,
    encoding="utf-8-sig",
    sep=","
)

todos_erros = []

todos_erros.extend(
    municipios_sem_dados
)

todos_erros.extend(
    municipios_com_erro
)

if len(todos_erros) > 0:

    df_erros = pd.DataFrame(
        todos_erros
    )

    df_erros.to_csv(
        ARQUIVO_ERROS,
        index=False,
        encoding="utf-8-sig"
    )

tempo_total = (
    time.time() - inicio
) / 60

print(
    "Arquivo:",
    ARQUIVO_FINAL
)

print(
    "Registros:",
    len(df_brasil)
)

print(
    "Municípios:",
    df_brasil["municipio_nome"].nunique()
)

print(
    "Estados:",
    df_brasil["UF"].nunique()
)

print(
    "Municípios sem dados:",
    len(municipios_sem_dados)
)

print(
    "Municípios com erro:",
    len(municipios_com_erro)
)

print(
    "Tempo total:",
    f"{tempo_total:.1f} minutos"
)

display(
    df_brasil.head(20)
)

files.download(
    ARQUIVO_FINAL
)

if len(todos_erros) > 0:
    files.download(
        ARQUIVO_ERROS
    )

TESTANDO CONEXÃO COM O INFODENGUE
Status da API: 200
OK! API do InfoDengue respondeu.

BAIXANDO MUNICÍPIOS DO IBGE

Total de municípios encontrados: 5570

Exemplo:


,geocode,municipio,UF
0,1100015,Alta Floresta D'Oeste,RO
1,1100023,Ariquemes,RO
2,1100031,Cabixi,RO
3,1100049,Cacoal,RO
4,1100056,Cerejeiras,RO
5,1100064,Colorado do Oeste,RO
6,1100072,Corumbiara,RO
7,1100080,Costa Marques,RO
8,1100098,Espigão D'Oeste,RO
9,1100106,Guajará-Mirim,RO



INICIANDO EXTRAÇÃO NACIONAL

Período: 2025 até 31/08/2026
Municípios: 5570
Consultas simultâneas: 8



Consultando municípios:   0%|          | 0/5570 [00:00<?, ?it/s]


CONSULTAS FINALIZADAS

Municípios com dados: 5570
Municípios sem dados: 0
Municípios com erro: 0

Unificando todas as cidades...

Duplicidades removidas: 0


EXTRAÇÃO CONCLUÍDA

Arquivo: Base_Dengue_Brasil_2025_ate_hoje.csv

Total de registros: 473,446

Municípios com registros: 5297

Estados: 27

Primeira data: 01/02/2026
Última data: 31/08/2025

Municípios sem dados: 0
Municípios com erro: 0

Tempo total: 10.7 minutos

AMOSTRA DA BASE FINAL:


...

,UF,registros
0,AC,1870
1,AL,8670
2,AM,5270
3,AP,1360
4,BA,35445
5,CE,15640
6,DF,85
7,ES,6630
8,GO,20910
9,MA,18445



Iniciando download do arquivo...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DOWNLOAD INICIADO!

Todos os municípios consultados retornaram sem erro registrado.
